# Índice de Caminabilidad — Piloto Av. Roosevelt
**ITT Cali Inteligente · Equipo de Gobierno de Datos**

Este notebook calcula las métricas de caminabilidad para el área de influencia
de la intervención en Av. Roosevelt, usando la red peatonal de OpenStreetMap
como línea base.

**Datos:** `data/itt_roosevelt/Roosevelt/Geojson_Roosevelt/`

**Instrucciones:**
1. Ejecutar las celdas en orden (Shift + Enter)
2. Los datos se cargan automáticamente desde la carpeta del proyecto
3. Los resultados y el mapa se generan automáticamente

## Celda 1 — Instalación de dependencias

In [ ]:
import subprocess, sys
def check_pkg(pkg):
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for p in ['osmnx', 'geopandas', 'matplotlib', 'pandas', 'numpy', 'folium', 'seaborn']:
    check_pkg(p)
print('Dependencias verificadas')

## Celda 2 — Importaciones y configuración

In [ ]:
import os, re, json, warnings, datetime
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import osmnx as ox
import matplotlib.pyplot as plt
import seaborn as sns
import folium
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#F4F6F9',
    'axes.facecolor': 'white',
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3
})
print('Configuración visual lista')

## Celda 3 — Parámetros, rutas y detección de entorno

In [ ]:
# Detectar entorno: Colab o local
if os.path.exists('/content'):
    # Google Colab: clonar el repo
    os.chdir('/content')
    import subprocess as _sp
    _sp.getoutput('rm -rf itt_repos_cali')
    print(_sp.getoutput('git clone -b jorge_itt https://github.com/j0rg3c45/Itt_repos_cali.git itt_repos_cali'))
    DATA_DIR = Path('/content/itt_repos_cali/data/itt_roosevelt/Roosevelt/Geojson_Roosevelt')
    IMG_DIR = Path('/content/itt_repos_cali/outputs/figures')
    RESULTS_DIR = Path('/content/itt_repos_cali/outputs/results')
else:
    # Ejecución local
    DATA_DIR = Path(os.getcwd()).parent / 'data' / 'itt_roosevelt' / 'Roosevelt' / 'Geojson_Roosevelt'
    IMG_DIR = Path(os.getcwd()).parent / 'outputs' / 'figures'
    RESULTS_DIR = Path(os.getcwd()).parent / 'outputs' / 'results'

IMG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Búsqueda flexible de archivos GeoJSON
def find_file(base_dir, pattern):
    regex = re.compile(pattern, re.IGNORECASE)
    for root, _dirs, files in os.walk(base_dir):
        for f in files:
            if regex.search(f) and f.lower().endswith('.geojson'):
                return os.path.join(root, f)
    return None

# Rutas a los archivos de datos
PATHS = {
    'poligono_buffer': find_file(DATA_DIR, r'Geojson_tramos_Roosevelt_Buffer_100'),
    'poligono_tramos': find_file(DATA_DIR, r'Geojson_tramos_Roosevelt\.geojson'),
    'siniestros':      find_file(DATA_DIR, r'BD_SINIESTROS'),
    'comparendos':     find_file(DATA_DIR, r'COMPARENDOS'),
    'homicidios':      find_file(DATA_DIR, r'HOMICIDIOS'),
    'hurtos':          find_file(DATA_DIR, r'HURTOS'),
    'sedes':           find_file(DATA_DIR, r'Sedes_educativas'),
    'vbg':             find_file(DATA_DIR, r'VBG'),
    'vif':             find_file(DATA_DIR, r'VIOLENCIA_INTRAFAMILIAR'),
}

# CRS Colombia
CRS_COLOMBIA = 3116
ZONA_NOMBRE = 'Av. Roosevelt — Cali'

print(f'DATA_DIR: {DATA_DIR}')
print(f'IMG_DIR:  {IMG_DIR}')
print(f'RESULTS_DIR: {RESULTS_DIR}')
print()
print('Archivos encontrados:')
for nombre, ruta in PATHS.items():
    ok = ruta is not None and os.path.exists(ruta)
    estado = chr(9989) if ok else chr(10060)
    print(f'  {estado}  {nombre:18s}: {Path(ruta).name if ruta else "NO ENCONTRADO"}')

## Celda 4 — Cargar polígono y calcular área

In [ ]:
# Cargar polígono de intervención (buffer 100m)
gdf_poligono = gpd.read_file(PATHS['poligono_buffer'])
polygon = gdf_poligono.geometry.iloc[0]

# Calcular área en sistema métrico (EPSG:3116 — Colombia)
gdf_m = gdf_poligono.to_crs(epsg=CRS_COLOMBIA)
area_m2 = gdf_m.geometry.area.iloc[0]
area_km2 = area_m2 / 1_000_000

print('=== ÁREA DE INTERVENCIÓN ===')
print(f'  Zona: {ZONA_NOMBRE}')
print(f'  Área: {area_m2:,.0f} m² ({area_m2/10000:.2f} ha)')
if 'BUFF_DIST' in gdf_poligono.columns:
    print(f'  Buffer aplicado: {gdf_poligono["BUFF_DIST"].iloc[0]:.0f} m a cada lado del eje')
if 'Shape_Leng' in gdf_poligono.columns:
    print(f'  Longitud aproximada del corredor: {gdf_poligono["Shape_Leng"].iloc[0]:.0f} m')

## Celda 5 — Descargar red peatonal desde OpenStreetMap

In [ ]:
print('Descargando red peatonal desde OpenStreetMap...')
G = ox.graph_from_polygon(polygon, network_type='walk')
print(f'Red descargada: {len(G.nodes)} nodos, {len(G.edges)} segmentos')

## Celda 6 — Calcular métricas de caminabilidad

In [ ]:
# Calcular estadísticas básicas de la red
stats = ox.basic_stats(G, area=area_m2)

longitud_total_km = stats['edge_length_total'] / 1000
densidad_km_km2 = longitud_total_km / area_km2

metricas_caminabilidad = {
    'Intersecciones peatonales': stats['intersection_count'],
    'Longitud red peatonal (km)': round(longitud_total_km, 2),
    'Longitud promedio segmento (m)': round(stats['edge_length_avg'], 1),
    'Densidad calle (km/km²)': round(densidad_km_km2, 2),
    'Nodos OSM': len(G.nodes),
    'Segmentos OSM': len(G.edges),
}

print('=== MÉTRICAS DE CAMINABILIDAD — LÍNEA BASE ===')
print(f'  Fecha de referencia: {datetime.date.today().isoformat()}')
print()
for k, v in metricas_caminabilidad.items():
    print(f'  {k}: {v}')

## Celda 7 — Cargar datos geoespaciales complementarios

In [ ]:
# Cargar todos los datasets disponibles
datasets = {}
for nombre, ruta in PATHS.items():
    if ruta and os.path.exists(ruta) and nombre not in ('poligono_buffer', 'poligono_tramos'):
        try:
            gdf = gpd.read_file(ruta)
            datasets[nombre] = gdf
            print(f'  {chr(9989)} {nombre:18s}: {len(gdf):>5} registros | Columnas: {list(gdf.columns)[:5]}...')
        except Exception as e:
            print(f'  {chr(10060)} {nombre:18s}: Error al cargar - {e}')

print(f'\nDatasets cargados: {len(datasets)}')

## Celda 8 — Mapa de la red peatonal

In [ ]:
fig, ax = ox.plot_graph(
    G,
    node_size=12,
    edge_linewidth=1.2,
    bgcolor='white',
    node_color='#2A9C8A',
    edge_color='#1A3A4A',
    figsize=(12, 12),
    show=False,
    close=False
)
ax.set_title(
    f'Red peatonal — {ZONA_NOMBRE}\nLínea base ITT · Cali Inteligente',
    fontsize=13, pad=15
)
plt.tight_layout()
plt.savefig(IMG_DIR / 'roosevelt_red_peatonal_linea_base.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Mapa guardado en: {IMG_DIR / "roosevelt_red_peatonal_linea_base.png"}')

## Celda 9 — Mapa interactivo con capas de datos

In [ ]:
# Centro del polígono para el mapa
centroid = polygon.centroid
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=15,
               tiles='CartoDB positron')

# Polígono de intervención
folium.GeoJson(
    gdf_poligono.to_crs('EPSG:4326'),
    name='Polígono intervención',
    style_function=lambda x: {'color': '#1B4F8A', 'fillColor': '#2E7D32', 'fillOpacity': 0.1, 'weight': 2}
).add_to(m)

# Colores por capa
COLORES = {
    'siniestros': 'red',
    'comparendos': 'orange',
    'homicidios': 'darkred',
    'hurtos': 'purple',
    'sedes': 'blue',
    'vbg': 'pink',
    'vif': 'darkpurple',
}

# Agregar capas de puntos
for nombre, gdf in datasets.items():
    if gdf.geometry.iloc[0].geom_type == 'Point':
        fg = folium.FeatureGroup(name=nombre.capitalize())
        color = COLORES.get(nombre, 'gray')
        for _, row in gdf.iterrows():
            folium.CircleMarker(
                location=[row.geometry.y, row.geometry.x],
                radius=3, color=color, fill=True, fill_opacity=0.6
            ).add_to(fg)
        fg.add_to(m)

folium.LayerControl().add_to(m)
m

## Celda 10 — Resumen de indicadores por dataset

In [ ]:
# Conteo de eventos por dataset dentro del polígono
print('=== INDICADORES COMPLEMENTARIOS — LÍNEA BASE ===')
print(f'  Zona: {ZONA_NOMBRE}')
print(f'  Fecha: {datetime.date.today().isoformat()}')
print()

indicadores = {}
for nombre, gdf in datasets.items():
    n_registros = len(gdf)
    densidad = n_registros / (area_m2 / 10000)  # por hectárea
    indicadores[nombre] = {'total': n_registros, 'densidad_por_ha': round(densidad, 2)}
    print(f'  {nombre:18s}: {n_registros:>5} registros | {densidad:.2f} por ha')

print()
print('Nota: estos valores constituyen la línea base para comparar')
print('después de ejecutadas las obras de intervención (2026-2028).')

## Celda 11 — Exportar resultados a CSV

In [ ]:
# Construir DataFrame de resultados
resultado = {
    'fecha_medicion': datetime.date.today().isoformat(),
    'poligono': 'Av. Roosevelt - Buffer 100m',
    'area_m2': round(area_m2, 0),
    'area_ha': round(area_m2 / 10000, 2),
    'intersecciones_peatonales': stats['intersection_count'],
    'longitud_red_peatonal_km': round(longitud_total_km, 2),
    'longitud_promedio_segmento_m': round(stats['edge_length_avg'], 1),
    'densidad_calle_km_km2': round(densidad_km_km2, 2),
    'nodos_osm': len(G.nodes),
    'segmentos_osm': len(G.edges),
    'fuente': 'OpenStreetMap via OSMnx',
    'momento': 'linea_base_pre_intervencion',
}

# Agregar conteos de datasets complementarios
for nombre, vals in indicadores.items():
    resultado[f'total_{nombre}'] = vals['total']
    resultado[f'densidad_{nombre}_por_ha'] = vals['densidad_por_ha']

df_resultados = pd.DataFrame([resultado])
output_csv = RESULTS_DIR / 'roosevelt_caminabilidad_linea_base.csv'
df_resultados.to_csv(output_csv, index=False)

print(f'Resultados exportados a: {output_csv}')
print()
print(df_resultados.T.to_string())

## Celda 12 — Descargar archivos (solo Google Colab)

In [ ]:
# Solo ejecutar en Google Colab
if os.path.exists('/content'):
    from google.colab import files
    files.download(str(output_csv))
    files.download(str(IMG_DIR / 'roosevelt_red_peatonal_linea_base.png'))
else:
    print('Ejecución local — archivos guardados en outputs/')